In [2]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from lifelines import WeibullAFTFitter
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('../utilities')
from preprocess import preprocess

PARTICIPANT_DATA_PATH = '../participant_data/'

In [3]:
# Define path to the single participant data folder.
PARTICIPANT_DATA_PATH = '../participant_data/'
SUBMISSION_DIR = 'my_prediction_submission_aft'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

In [4]:
# Define all 16 event pairs
index_events = ["Borrow", "Deposit", "Repay", "Withdraw"]
outcome_events = index_events + ["Liquidated"]
event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing and Predicting for: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- Load and Preprocess ---
    try:
        train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
        test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))
    except FileNotFoundError as e:
        print(f"Data not found for {dataset_path}. Skipping.")
        continue
        
    X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

    # --- Initally Train Model ---
    try:
        lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        lifelines_train_df = lifelines_train_df.loc[lifelines_train_df['timeDiff'] > 0].copy()

        ##-- START: NEW CODE ADDED ---##
        print("Training model on initial training data...")
        model1 = WeibullAFTFitter(penalizer=0.1)
        model1.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')
        
        summary_df = model1.summary

        hr = summary_df['exp(coef)'].to_list() # hazard ratio
        p_values = summary_df['p'].to_list() # p values

        cols_to_keep = []
        for index, row in summary_df.iterrows():
            if row['p'] <= 0.00001: # means only features that change hazard by less than ±20% are dropped
                cols_to_keep.append(index)
        
        col_names = [name for _, name in cols_to_keep]
        cols_to_keep.pop()
        cols_to_keep.pop()
        col_names = [name for _, name in cols_to_keep]

        new_train_df = lifelines_train_df[col_names + ['timeDiff', 'status']]

        ##-- END: NEW CODE ADDED ---#

        # --- Train With the Selected Features ---
        model2 = WeibullAFTFitter(penalizer=0.1)
        print("Training model on selected training data...")
        model2.fit(new_train_df, duration_col='timeDiff', event_col='status')

        # --- Finaly Generate and Save Predictions ---
        print(f"Generating predictions for {dataset_path}...")
        predictions = model2.predict_median(X_test_processed)
        
        # Save predictions to a CSV file
        prediction_filename = dataset_path.replace(os.sep, '_') + '.csv'
        prediction_save_path = os.path.join(SUBMISSION_DIR, prediction_filename)
        pd.DataFrame(predictions).to_csv(prediction_save_path, header=False, index=False)
        print(f"  -> Predictions saved to {prediction_save_path}")
        
    except (ConvergenceError, ValueError) as e:
        print(f"\nERROR: The model for {dataset_path} failed to train. No prediction file will be created.")
        print(f"Details: {e}")

print("\n\nAll prediction files have been generated.")


Processing and Predicting for: Borrow -> Deposit
Training model on initial training data...
Training model on selected training data...
Generating predictions for Borrow/Deposit...
  -> Predictions saved to my_prediction_submission_aft/Borrow_Deposit.csv

Processing and Predicting for: Borrow -> Repay
Training model on initial training data...
Training model on selected training data...
Generating predictions for Borrow/Repay...
  -> Predictions saved to my_prediction_submission_aft/Borrow_Repay.csv

Processing and Predicting for: Borrow -> Withdraw
Training model on initial training data...
Training model on selected training data...
Generating predictions for Borrow/Withdraw...
  -> Predictions saved to my_prediction_submission_aft/Borrow_Withdraw.csv

Processing and Predicting for: Borrow -> Liquidated
Training model on initial training data...
Training model on selected training data...
Generating predictions for Borrow/Liquidated...
  -> Predictions saved to my_prediction_submiss

In [5]:
output_zip_filename = 'submission_aft'
shutil.make_archive(output_zip_filename, 'zip', SUBMISSION_DIR)

print(f"Successfully created '{output_zip_filename}.zip' from the '{SUBMISSION_DIR}' directory.")
print("You can now upload this file to the CodaBench competition.")

Successfully created 'submission_aft.zip' from the 'my_prediction_submission_aft' directory.
You can now upload this file to the CodaBench competition.
